# HD 189733 b Transit Photometry

**Author:** Biswajit Jana

This notebook is a small hands-on project for looking at the transit of **HD 189733 b** using public TESS data.  
The idea is simple: download the light curve, fold it using the known orbital period, and estimate how much light the planet blocks.

It is written for Google Colab and for people who want to try exoplanet photometry without setting up a full research pipeline.

## 1. Install the packages

In [ ]:
!pip -q install lightkurve pandas numpy matplotlib astropy

## 2. Set up the notebook

In [ ]:
from pathlib import Path
from urllib.parse import quote
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import astropy.units as u
import lightkurve as lk

TARGET = "HD 189733"
PLANET = "HD 189733 b"

MAX_PRODUCTS = 2
BIN_MINUTES = 10

OUTDIR = Path("outputs")
OUTDIR.mkdir(exist_ok=True)

plt.rcParams.update({
    "figure.figsize": (10, 5),
    "axes.grid": True,
    "font.size": 11
})

## 3. Get the published planet values

I use the NASA Exoplanet Archive values for the orbital period, transit midpoint, duration, and radius ratio.  
These are used to fold the data at the expected transit time.

In [ ]:
def query_planet(planet_name):
    base = "https://exoplanetarchive.ipac.caltech.edu/TAP/sync"
    sql = f"""
    SELECT pl_name, hostname, pl_orbper, pl_tranmid, pl_trandur,
           pl_ratror, pl_rade, pl_radj, st_rad, st_teff, sy_dist
    FROM pscomppars
    WHERE pl_name = '{planet_name}'
    """
    return pd.read_csv(f"{base}?query={quote(sql)}&format=csv")

props = query_planet(PLANET).iloc[0]

PERIOD = float(props["pl_orbper"])
T0_BJD = float(props["pl_tranmid"])
DURATION_HR = float(props["pl_trandur"])
RP_RS = float(props["pl_ratror"]) if pd.notna(props["pl_ratror"]) else np.nan
EXPECTED_DEPTH_PERCENT = (RP_RS**2 * 100) if np.isfinite(RP_RS) else np.nan

target_info = pd.DataFrame({
    "quantity": ["Period [days]", "Transit midpoint [BJD]", "Duration [hours]", "Rp/Rs", "Expected depth [%]"],
    "value": [PERIOD, T0_BJD, DURATION_HR, RP_RS, EXPECTED_DEPTH_PERCENT]
})

display(target_info)
target_info.to_csv(OUTDIR / "target_info.csv", index=False)

## 4. Download the TESS light curve

To keep the notebook quick, I only download the first couple of available products.  
That is enough for a first demonstration and avoids making Colab slow.

In [ ]:
search = lk.search_lightcurve(
    TARGET,
    mission="TESS",
    author="SPOC",
    exptime="short"
)

if len(search) == 0:
    warnings.warn("No SPOC short-cadence product found. Trying a broader TESS search.")
    search = lk.search_lightcurve(TARGET, mission="TESS")

print(search[:10])

collection = search[:MAX_PRODUCTS].download_all(quality_bitmask="default")

light_curves = []
for item in collection:
    try:
        item = item.select_flux("pdcsap_flux")
    except Exception:
        pass

    item = item.remove_nans().normalize()
    light_curves.append(item)

lc = lk.LightCurveCollection(light_curves).stitch().remove_nans().normalize()
print(lc)

## 5. Look at the raw light curve

This first plot is just a check.  
The raw light curve can still include stellar activity, instrumental trends, and sector-to-sector changes.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

ax.scatter(lc.time.value, lc.flux.value, s=4, alpha=0.45, linewidths=0)
ax.set_title(f"{TARGET}: TESS light curve")
ax.set_xlabel("Time [mission days]")
ax.set_ylabel("Normalised flux")

lo, hi = np.nanpercentile(lc.flux.value, [1, 99])
ax.set_ylim(lo - 0.002, hi + 0.002)

fig.tight_layout()
fig.savefig(OUTDIR / "01_tess_light_curve.png", dpi=250)
plt.show()

## 6. Put the archive transit time into the TESS time system

The archive midpoint is normally given as a full BJD.  
TESS light curves use a shifted time system, so this helper moves the transit time into the same time range as the data.

In [ ]:
def convert_epoch(t0_bjd, time_values, period):
    median_t = np.nanmedian(time_values)

    candidates = {
        "BJD": t0_bjd,
        "BTJD": t0_bjd - 2457000,
        "BKJD": t0_bjd - 2454833,
    }

    label, base_t0 = min(candidates.items(), key=lambda item: abs(item[1] - median_t))
    n_shift = np.round((median_t - base_t0) / period)
    epoch_time = base_t0 + n_shift * period

    return epoch_time, label, int(n_shift)

EPOCH_TIME, TIME_SYSTEM, N_SHIFT = convert_epoch(T0_BJD, lc.time.value, PERIOD)

print("Time system used:", TIME_SYSTEM)
print("Shifted by this many orbits:", N_SHIFT)
print("Transit midpoint used in this notebook:", EPOCH_TIME)

## 7. Remove slow trends without removing the transit

Before flattening the data, I mask the expected transit region.  
That stops the trend-removal step from treating the transit dip as a trend.

In [ ]:
time = lc.time.value
duration_days = DURATION_HR / 24

phase_days = ((time - EPOCH_TIME + 0.5 * PERIOD) % PERIOD) - 0.5 * PERIOD
transit_mask = np.abs(phase_days) < 1.5 * duration_days

flat_lc = (
    lc.flatten(
        window_length=301,
        polyorder=2,
        mask=transit_mask,
        niters=2,
        sigma=5
    )
    .remove_outliers(sigma=5)
    .remove_nans()
    .normalize()
)

fig, ax = plt.subplots(figsize=(10, 4))

ax.scatter(flat_lc.time.value, flat_lc.flux.value, s=4, alpha=0.45, linewidths=0)
ax.set_title(f"{TARGET}: detrended light curve")
ax.set_xlabel("Time [mission days]")
ax.set_ylabel("Normalised flux")

lo, hi = np.nanpercentile(flat_lc.flux.value, [1, 99])
ax.set_ylim(lo - 0.002, hi + 0.002)

fig.tight_layout()
fig.savefig(OUTDIR / "02_detrended_light_curve.png", dpi=250)
plt.show()

## 8. Fold the light curve and estimate the transit depth

The depth is estimated from the median in-transit and out-of-transit flux:

\[
\delta = 1 - \frac{F_\mathrm{in}}{F_\mathrm{out}}
\]

This is only a simple first estimate, but it is enough to show the transit clearly.

In [ ]:
folded = flat_lc.fold(period=PERIOD, epoch_time=EPOCH_TIME)
binned = folded.bin(time_bin_size=BIN_MINUTES * u.minute)

phase_hr = folded.time.value * 24
flux = folded.flux.value

bin_phase_hr = binned.time.value * 24
bin_flux = binned.flux.value

in_transit = np.abs(phase_hr) < DURATION_HR / 2
out_transit = (np.abs(phase_hr) > 1.5 * DURATION_HR) & (np.abs(phase_hr) < 4.5)

f_in = np.nanmedian(flux[in_transit])
f_out = np.nanmedian(flux[out_transit])

depth_fraction = 1 - f_in / f_out
depth_percent = depth_fraction * 100
depth_ppm = depth_fraction * 1e6

summary = pd.DataFrame([{
    "Target": TARGET,
    "Planet": PLANET,
    "Period_days": PERIOD,
    "Duration_hours": DURATION_HR,
    "Measured_depth_percent": depth_percent,
    "Measured_depth_ppm": depth_ppm,
    "Archive_Rp_Rs": RP_RS,
    "Expected_depth_percent": EXPECTED_DEPTH_PERCENT,
    "TESS_products_used": MAX_PRODUCTS
}])

display(summary)
summary.to_csv(OUTDIR / "transit_depth_summary.csv", index=False)

## 9. Final plot

This is the main figure from the notebook.

In [ ]:
zoom = np.abs(phase_hr) < 5
zoom_bin = np.abs(bin_phase_hr) < 5

fig, ax = plt.subplots(figsize=(9, 5.5))

ax.scatter(
    phase_hr[zoom],
    flux[zoom],
    s=5,
    alpha=0.20,
    linewidths=0,
    label="Folded TESS data"
)

ax.plot(
    bin_phase_hr[zoom_bin],
    bin_flux[zoom_bin],
    "o-",
    linewidth=2,
    markersize=4,
    label=f"{BIN_MINUTES}-min bins"
)

ax.axvline(0, linestyle="--", linewidth=1, alpha=0.7)
ax.axhline(1, linestyle=":", linewidth=1, alpha=0.7)

ax.set_xlim(-5, 5)
ax.set_ylim(0.965, 1.012)

ax.set_title(
    f"{PLANET}: folded TESS transit\n"
    f"Depth ≈ {depth_percent:.3f}% ({depth_ppm:.0f} ppm), "
    f"expected ≈ {EXPECTED_DEPTH_PERCENT:.3f}%"
)
ax.set_xlabel("Time from mid-transit [hours]")
ax.set_ylabel("Normalised flux")
ax.legend()

fig.tight_layout()
fig.savefig(OUTDIR / "03_hd189733b_transit.png", dpi=300)
plt.show()

## 10. Save the results

In [ ]:
import shutil

zip_path = shutil.make_archive("hd189733b_transit_results", "zip", OUTDIR)
print("Created:", zip_path)